In [ ]:
import requests
import json
import math
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.features import (
    shot_distance,
    shot_angle,
    extract_freezefeature,
    extract_pass_features,
)

season_ids = [90, 4, 1, 42]
train_shots = []
test_shots = []

for season_id in season_ids:

    url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/11/{season_id}.json"

    response = requests.get(url)
    matches = response.json()
    sorted_matches = sorted(matches, key=lambda x: x["match_week"])
    match_ids = [match["match_id"] for match in sorted_matches]
    match_weeks = [match["match_week"] for match in sorted_matches]
    
    for match_id in match_ids:
        url_match = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
        response_match = requests.get(url_match)
        events = response_match.json()

        for event in events:
            if event["type"]["name"] == "Shot" and event["team"]["name"] == "Barcelona":
                x_loc1, y_loc1 = event["location"]
                dist = round(shot_distance(x_loc1, y_loc1), 2)
                angle = round(shot_angle(x_loc1, y_loc1), 0)
                ff_feats = extract_freezefeature(event)
                pass_id = event["shot"].get("key_pass_id", None)

                shot = {
                    "week": next(m["match_week"] for m in sorted_matches if m["match_id"] == match_id),
                    "time": f"{str(event['minute']).zfill(2)}:{str(event['second']).zfill(2)}",
                    "player": event["player"]["name"],
                    "distance": dist,
                    "angle": angle,
                    "type": event["shot"]["type"]["name"],
                    "technique": event["shot"]["technique"]["name"],
                    "outcome": event["shot"]["outcome"]["name"],
                    "goal/no goal": 1 if event["shot"]["outcome"]["name"] == "Goal" else 0,
                    "type_b": 0 if event["shot"]["type"]["name"] == "Open Play" else 1,
                    "technique_b": 0 if event["shot"]["technique"]["name"] == "Normal" else 1,
                    "distance_d": dist / 127,
                    "distance_l": math.log1p(dist),
                    "angle_d": angle / 180,
                    "n_def_1_5": ff_feats.get("n_def_1_5"),
                    "n_def_3_0": ff_feats.get("n_def_3_0"),
                    "dist_nearest_def": ff_feats.get("dist_nearest_def"),
                    "gk_dist_to_shooter": ff_feats.get("gk_dist_to_shooter"),
                    "free_kick_flag": 1 if event["shot"]["type"]["name"] == "Free Kick" else 0,
                    "penalty_flag": 1 if event["shot"]["type"]["name"] == "Penalty" else 0,
                    "pass_id": pass_id
                }
                
                pass_feats = extract_pass_features(event, events, pass_id)
                shot.update(pass_feats)
                
                if season_id == 42:
                    shot["statsbomb_xg"] = event["shot"]["statsbomb_xg"]
                    test_shots.append(shot)
                else:
                    train_shots.append(shot)

print(f"Total Barcelona training shots collected: {len(train_shots)}")
print(f"Total Barcelona testing shots collected: {len(test_shots)}")
df_train = pd.DataFrame(train_shots)
df_test = pd.DataFrame(test_shots)

df_train.insert(0, "shot_id", range(1, len(df_train) + 1))
df_test.insert(0, "shot_id", range(1, len(df_test) + 1))

df_train.to_excel("training_barcelona_shots.xlsx", index=False)
df_test.to_excel("testing_barcelona_shots.xlsx", index=False)